In [2]:
import pandas as pd
import os
from pathlib import Path

In [3]:
PROJECT_ROOT = Path.cwd().parent

In [33]:
def check_bt_ready(df):
    problems = []

    # 1. Check index type
    if not isinstance(df.index, pd.DatetimeIndex):
        problems.append('Index is not a DatetimeIndex')
    
    # 2. Check sort order
    if not df.index.is_monotonic_increasing:
        problems.append('Index is not sorted (oldest to newest)')
    
    # 3. Check for required columns
    # Note: Backtrader defaults to lowercase (open, high, etc.)
    required = ['open', 'high', 'low', 'close', 'volume']
    missing = [col for col in required if col not in df.columns]
    
    if missing:
        problems.append(f'The df is missing: {missing}')
    else:
        # 4. Check for NaNs and Data Types (Only if columns exist to avoid KeyError)
        if df[required].isnull().values.any():
            problems.append('Contains NaN values')
            
        non_numeric = df[required].select_dtypes(exclude=['number']).columns.tolist()
        if non_numeric:
            problems.append(f'Non-numeric data found in: {non_numeric}')
    
    # 5. Length check
    if len(df) != 1200:
        problems.append(f'df length is {len(df)}, expected 1200')
    
    return problems

In [ ]:
data_dir = Path(f"{PROJECT_ROOT}/data/raw")
files = list(data_dir.glob("*"))

for file in files:
    df = pd.read_csv(file, index_col=0, parse_dates=True)
    
    problems = check_bt_ready(df)
    if problems: print(f'{str(file)[len(str(PROJECT_ROOT))+10:]} Has problem: {problems}')

# If there is not output the data is ready for backtrader
